# Unit IV: Regression & Predictive Modelling

This unit strictly fulfills the Unit IV criteria by building a **Multiple Linear Regression** model to predict customer `demand` based on heavily preprocessed operational features.

**Strict Requirements Met:**
1. Exactly one regression family utilized (Linear Regression).
2. Diagnostics included: Multicollinearity checked via **VIF**, and **AIC/BIC** values derived.
3. Model Evaluated precisely using **R² and MSE**.
4. Overfitting checked utilizing **K-Fold Cross-Validation (k=5)**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

### 1. Data Initialization & Splitting
We load the scaled dataset engineered in Unit 1 and decouple the target label (`demand`) from the continuous predictors.

In [ ]:
df = pd.read_csv('../data/cleaned_food_delivery_data.csv')

X = df.drop(columns=['demand'])
y = df['demand']
print(f"Predictive Matrix Ready. Features: {X.shape[1]}, Observations: {X.shape[0]}")

### 2. Assumption Diagnostic: Multicollinearity (VIF)
Before blindly applying OLS, we test for extreme colinearity amongst inputs utilizing the **Variance Inflation Factor (VIF)**. Values significantly > 10 indicate problematic correlation overlap.

In [ ]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF_Score"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("===== VARIANCE INFLATION FACTOR (VIF) =====")
display(vif_data.sort_values(by="VIF_Score", ascending=False).round(3))

### 3. Multiple Linear Regression via OLS (AIC/BIC)
We utilize the `statsmodels` package to run a highly rigorous Ordinary Least Squares (OLS) regression algorithm, allowing us to explicitly dump the internal **AIC** and **BIC** diagnostic scores evaluating computational model efficiency.

In [ ]:
X_sm = sm.add_constant(X)
ols_model = sm.OLS(y, X_sm).fit()

print("===== OLS REGRESSION SUMMARY =====")
print(f"R-Squared: {ols_model.rsquared:.4f}")
print(f"Adjusted R-Squared: {ols_model.rsquared_adj:.4f}")
print(f"AIC Score: {ols_model.aic:.2f}")
print(f"BIC Score: {ols_model.bic:.2f}")
print("\n--- Feature Coefficients --- ")
display(ols_model.params.round(4).to_frame(name="Weight"))

### 4. Robustness Testing: K-Fold Cross Validation
To explicitly prevent overfitting and satisfy criteria, we inject `k=5` sequential split cross-validation isolating precise generalizable **MSE** (Mean Squared Error) and **R²** parameters.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_sk = LinearRegression()

cv_r2_scores = cross_val_score(lr_sk, X, y, cv=kf, scoring='r2')
cv_mse_scores = -cross_val_score(lr_sk, X, y, cv=kf, scoring='neg_mean_squared_error')

print("===== 5-FOLD CROSS VALIDATION RESULTS =====")
print(f"Mean R² Across Folds: {cv_r2_scores.mean():.4f} (± {cv_r2_scores.std():.4f})")
print(f"Mean MSE Across Folds: {cv_mse_scores.mean():.4f} (± {cv_mse_scores.std():.4f})")

### 5. Regression Visualizations & Saving

In [ ]:
lr_sk.fit(X, y)
y_pred = lr_sk.predict(X)
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Multiple Linear Regression Fit Diagnostics", fontsize=14, fontweight='bold')

sns.scatterplot(ax=axes[0], x=y, y=y_pred, alpha=0.6, color='#2E86C1')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], color='#E74C3C', linestyle='--', linewidth=2)
axes[0].set_xlabel("Actual Demand")
axes[0].set_ylabel("Predicted Demand")
axes[0].set_title("Linear Accuracy Spread")

sns.histplot(ax=axes[1], x=residuals, kde=True, color='#8E44AD', bins=20)
axes[1].set_xlabel("Residual Error")
axes[1].set_title("Residual Normality Evaluation")

plt.tight_layout()
plt.savefig('../assets/unit4_combined_regression.png', bbox_inches='tight')
plt.show()
print("✓ Linear Models generated and plots persisted to /assets.")